In [1]:
import requests
import csv

API_KEY = "TwBLHAAyy9Ec86yp5x5nWVcZu9XKozTn"
BASE_URL = "https://api.dune.com/api/v1/uploads"
OUTPUT_FILE = "../../data/csv_files/uploads_tables.csv"

def human_readable_bytes(num_bytes, decimals=2):
    """Convert bytes to KB/MB/GB/TB."""
    num_bytes = int(num_bytes)
    if num_bytes == 0:
        return "0 B"
    units = ["B", "KB", "MB", "GB", "TB"]
    idx = 0
    size = float(num_bytes)
    while size >= 1024 and idx < len(units) - 1:
        size /= 1024.0
        idx += 1
    return f"{size:.{decimals}f} {units[idx]}"

def fetch_all_uploads():
    all_tables = []
    limit = 10000  # max allowed
    offset = 0

    while True:
        url = f"{BASE_URL}?api_key={API_KEY}&limit={limit}&offset={offset}"
        print(f"Fetching: offset {offset}")

        resp = requests.get(url)
        resp.raise_for_status()
        data = resp.json()

        tables = data.get("tables", [])
        if not tables:
            break  # no more pages

        all_tables.extend(tables)
        offset += limit

    return all_tables


def save_to_csv(tables):
    with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["full_name", "is_private", "table_size_bytes", "table_size_human"])

        for t in tables:
            size_bytes = t.get("table_size_bytes", 0)
            human_size = human_readable_bytes(size_bytes)

            writer.writerow([
                t.get("full_name"),
                t.get("is_private"),
                size_bytes,
                human_size
            ])

    print(f"\nCSV created: {OUTPUT_FILE}")


if __name__ == "__main__":
    tables = fetch_all_uploads()
    save_to_csv(tables)


Fetching: offset 0
Fetching: offset 10000

CSV created: uploads_tables.csv


In [2]:
import requests
import csv

API_KEY = "TwBLHAAyy9Ec86yp5x5nWVcZu9XKozTn"
BASE_URL = "https://api.dune.com/api/v1/uploads"
OUTPUT_FILE = "../../data/csv_files/uploads_tables_sorted.csv"

def human_readable_bytes(num_bytes, decimals=2):
    """Convert bytes to KB/MB/GB/TB."""
    num_bytes = int(num_bytes)
    if num_bytes == 0:
        return "0 B"
    units = ["B", "KB", "MB", "GB", "TB"]
    idx = 0
    size = float(num_bytes)
    while size >= 1024 and idx < len(units)-1:
        size /= 1024.0
        idx += 1
    return f"{size:.{decimals}f} {units[idx]}"

def fetch_all_uploads():
    all_tables = []
    limit = 10000  # max allowed by Dune
    offset = 0

    while True:
        url = f"{BASE_URL}?api_key={API_KEY}&limit={limit}&offset={offset}"
        print(f"Fetching: offset={offset}")

        resp = requests.get(url)
        resp.raise_for_status()
        data = resp.json()

        tables = data.get("tables", [])
        if not tables:
            break  # reached end of dataset

        all_tables.extend(tables)
        offset += limit

    return all_tables

def save_sorted_csv(tables):
    # Convert sizes to integer and sort descending
    for t in tables:
        t["table_size_bytes"] = int(t.get("table_size_bytes", 0))

    tables_sorted = sorted(tables, key=lambda x: x["table_size_bytes"], reverse=True)

    # Write sorted CSV
    with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["full_name", "is_private", "table_size_bytes", "table_size_human"])

        for t in tables_sorted:
            size_bytes = t["table_size_bytes"]
            human_size = human_readable_bytes(size_bytes)

            writer.writerow([
                t.get("full_name"),
                t.get("is_private"),
                size_bytes,
                human_size
            ])

    print(f"\nSorted CSV created → {OUTPUT_FILE}")

if __name__ == "__main__":
    tables = fetch_all_uploads()
    save_sorted_csv(tables)


Fetching: offset=0
Fetching: offset=10000

Sorted CSV created → uploads_tables_sorted.csv
